In [3]:
# Imports & purpose (COMMENTED)
# Synthetic NHS-style Clinical Dataset Generator
# ---------------------------------------------
# This notebook creates a fake (synthetic) dataset
# that looks like NHS clinical data.
# We will intentionally add data quality issues
# so that later we can detect and monitor them.
#
# IMPORTANT:
# - No real patient data is used
# - Safe for GitHub & Medium
# ---------------------------------------------

# pandas: used for tabular data (DataFrame)
import pandas as pd

# numpy: used for random number generation
import numpy as np

# datetime utilities for dates
from datetime import datetime, timedelta

In [7]:
# Set random seed so results are reproducible
# (important for debugging and for portfolio reviewers)
np.random.seed(42)

# Number of patient records to generate
n_records = 1000
# (important for debugging and for portfolio reviewers)
np.random.seed(42)

# Number of patient records to generate
n_records = 1000

In [8]:
# Create unique patient IDs
# Example IDs: 100000, 100001, 100002, ...
patient_ids = np.arange(100000, 100000 + n_records)


In [9]:
# Generate random admission dates within 2023
admission_dates = [
    # Start from 1 Jan 2023 and add a random number of days
    datetime(2023, 1, 1) + timedelta(days=np.random.randint(0, 365))
    for _ in range(n_records)
]

In [11]:
# Generate length of stay between 1 and 14 days
length_of_stay = np.random.randint(1, 15, size=n_records)

# Discharge date = admission date + length of stay
discharge_dates = [
    admission_dates[i] + timedelta(days=int(length_of_stay[i]))
    for i in range(n_records)
]

In [12]:
# Generate patient ages between 0 and 99
ages = np.random.randint(0, 100, size=n_records)

# Generate gender values
#  intentionally include:
# - "Unknown"
# - None (missing values)
genders = np.random.choice(
    ["Male", "Female", "Unknown", None],
    size=n_records,
    p=[0.48, 0.48, 0.02, 0.02]
)

# Generate diagnosis codes (ICD-10 style)
# We intentionally include None to simulate missing diagnosis codes
diagnosis_codes = np.random.choice(
    ["I10", "E11", "J45", "C34", None],
    size=n_records,
    p=[0.3, 0.3, 0.2, 0.15, 0.05]
)

In [13]:
# Combine all generated fields into a single DataFrame
df = pd.DataFrame({
    "patient_id": patient_ids,
    "admission_date": admission_dates,
    "discharge_date": discharge_dates,
    "age": ages,
    "gender": genders,
    "primary_diagnosis_code": diagnosis_codes
})

# Preview the first 5 rows
df.head()

,patient_id,admission_date,discharge_date,age,gender,primary_diagnosis_code
0,100000,2023-04-13,2023-04-21,61,Female,C34
1,100001,2023-12-15,2023-12-23,9,Male,None
2,100002,2023-09-28,2023-10-09,20,Female,E11
3,100003,2023-04-17,2023-04-30,35,Male,E11
4,100004,2023-03-13,2023-03-14,11,Female,E11


In [15]:
# ---------------------------------------------
# INTENTIONAL DATA QUALITY ISSUES
# ---------------------------------------------

# Introduce missing age values (completeness issue)
df.loc[np.random.choice(df.index, 50), "age"] = None

In [16]:
# Introduce invalid age values (validity issue)
# Negative ages should never exist
df.loc[np.random.choice(df.index, 20), "age"] = -5

In [17]:
# Introduce date consistency errors
# Discharge date earlier than admission date
df.loc[np.random.choice(df.index, 15), "discharge_date"] = (
    df["admission_date"] - timedelta(days=1)
)

In [18]:

# Introduce duplicate records
# Sample 10 existing rows and append them
duplicates = df.sample(10)
df = pd.concat([df, duplicates], ignore_index=True)

# Check final shape after duplicates
df.shape

(1010, 6)

In [27]:
# Save the dataset to the data/raw folder
# ../ means "go up one directory from notebooks/"
output_path = "../data/raw/clinical_data.csv"

df.to_csv(output_path, index=False)

# Show where the file was saved
output_path

'../data/raw/clinical_data.csv'